# Deep Dataset EDA

Run after `prepare_dataset.py` and `prepare_drink_waste.py`. Catches training-breaking issues before you spend 30+ min on a bad run.

Sections:
1. Setup
2. Image counts per split
3. **File integrity** — every image loads, not corrupt, not zero-byte
4. **Orphans** — images without labels, labels without images
5. **Cross-split duplicates** — exact-content dupes across splits (data leak)
6. Class distribution + per-split balance
7. Source distribution (TACO vs Drink Waste) per split
8. Image dimensions + aspect ratios
9. Image color modes (grayscale sneaking in?)
10. Bbox size distribution
11. Bbox aspect ratios (extreme shapes hurt training)
12. Per-class bbox stats table
13. Bbox center-position heatmap (per class)
14. Boxes per image
15. Class co-occurrence matrix
16. Sample gallery per class with boxes drawn
17. Final health checks (label format)

In [ ]:
# --- 1. setup ---
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ML_DIR = Path.cwd()
if ML_DIR.name == 'notebooks':
    ML_DIR = ML_DIR.parent
DATA = ML_DIR / 'data'
CLASS_NAMES = ['bottle', 'cup', 'can', 'wrapper', 'paper']
SPLITS = ('train', 'val', 'test')
IMG_EXTS = ('.jpg', '.jpeg', '.png')
assert DATA.exists(), f'missing {DATA}. Run the prep scripts first.'

def imgs_in(split):
    return [p for p in (DATA / 'images' / split).glob('*') if p.suffix.lower() in IMG_EXTS]

def label_for(img_path):
    return DATA / 'labels' / img_path.parent.name / (img_path.stem + '.txt')

def parse_label(path):
    if not path.exists():
        return []
    out = []
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) == 5:
            try:
                out.append((int(parts[0]), *map(float, parts[1:])))
            except ValueError:
                continue
    return out

print('ml-training dir:', ML_DIR)

## 2. Image counts per split

In [ ]:
counts = {s: len(imgs_in(s)) for s in SPLITS}
total = sum(counts.values())
for s in SPLITS:
    pct = (counts[s] / total * 100) if total else 0
    print(f'{s:>5}: {counts[s]:>6} images ({pct:.1f}%)')
print(f'{"total":>5}: {total:>6}')

## 3. File integrity

Every image file must open without error. Corrupt/truncated JPEGs cause Ultralytics to either skip (best case) or crash mid-training (worst case). Also flags zero-byte files.

In [ ]:
corrupt, empty = [], []
for s in SPLITS:
    for p in imgs_in(s):
        if p.stat().st_size == 0:
            empty.append(p)
            continue
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception as e:
            corrupt.append((p, str(e)))
print(f'zero-byte: {len(empty)}')
for p in empty[:5]:
    print(f'  {p.relative_to(DATA)}')
print(f'corrupt:   {len(corrupt)}')
for p, e in corrupt[:5]:
    print(f'  {p.relative_to(DATA)}: {e}')
if not empty and not corrupt:
    print('\nno integrity issues')

## 4. Orphans

- Images without label files: Ultralytics will treat these as no-object (background) — usually harmless but a signal the prep failed on some images.
- Label files without images: leftover from old runs, will be silently ignored.

In [ ]:
orphan_images, orphan_labels = defaultdict(list), defaultdict(list)
for s in SPLITS:
    img_stems = {p.stem for p in imgs_in(s)}
    lbl_stems = {p.stem for p in (DATA / 'labels' / s).glob('*.txt')}
    for stem in img_stems - lbl_stems:
        orphan_images[s].append(stem)
    for stem in lbl_stems - img_stems:
        orphan_labels[s].append(stem)
    print(f'{s}: {len(orphan_images[s])} images without labels, {len(orphan_labels[s])} labels without images')

## 5. Cross-split duplicates (data leak check)

If the same image ends up in both train and test, test mAP will be overstated. MD5 hash every image, flag collisions across splits.

In [ ]:
def md5(path, chunk=65536):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

hash_to_locs = defaultdict(list)
for s in SPLITS:
    for p in imgs_in(s):
        hash_to_locs[md5(p)].append((s, p.name))

same_split_dupes = 0
cross_split_dupes = []
for h, locs in hash_to_locs.items():
    if len(locs) < 2:
        continue
    splits_involved = {s for s, _ in locs}
    if len(splits_involved) == 1:
        same_split_dupes += len(locs) - 1
    else:
        cross_split_dupes.append(locs)

print(f'same-split duplicates: {same_split_dupes}')
print(f'CROSS-SPLIT duplicates (data leak): {len(cross_split_dupes)}')
for locs in cross_split_dupes[:10]:
    print(f'  {locs}')

## 6. Class distribution + per-split balance

Watch for:
- A class with 0 instances anywhere → remap bug.
- Imbalance >5:1 in train → consider oversampling the minority.
- Per-split ratio drift: if `wrapper` is 40% of train but 5% of test, test mAP won't generalize.

In [ ]:
dist = {s: Counter() for s in SPLITS}
for s in SPLITS:
    for lbl in (DATA / 'labels' / s).glob('*.txt'):
        for cls, *_ in parse_label(lbl):
            dist[s][cls] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
width = 0.25
x = list(range(len(CLASS_NAMES)))
for i, s in enumerate(SPLITS):
    axes[0].bar([v + (i - 1) * width for v in x], [dist[s].get(c, 0) for c in range(len(CLASS_NAMES))], width, label=s)
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASS_NAMES)
axes[0].set_ylabel('instances')
axes[0].set_title('absolute class counts per split')
axes[0].legend()

# normalize per-split to fractions for comparability
for i, s in enumerate(SPLITS):
    tot = sum(dist[s].values()) or 1
    axes[1].bar([v + (i - 1) * width for v in x], [dist[s].get(c, 0) / tot for c in range(len(CLASS_NAMES))], width, label=s)
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASS_NAMES)
axes[1].set_ylabel('fraction of split')
axes[1].set_title('class share per split (should be similar across splits)')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"{'class':<10} {'train':>7} {'val':>7} {'test':>7} {'total':>7}  {'train%':>7} {'test%':>7}")
for c, n in enumerate(CLASS_NAMES):
    t, v, te = dist['train'].get(c, 0), dist['val'].get(c, 0), dist['test'].get(c, 0)
    tt, tot_te = sum(dist['train'].values()) or 1, sum(dist['test'].values()) or 1
    print(f'{n:<10} {t:>7} {v:>7} {te:>7} {t+v+te:>7}  {t/tt*100:>6.1f}% {te/tot_te*100:>6.1f}%')

train_totals = list(dist['train'].values())
if train_totals and min(train_totals) > 0:
    ratio = max(train_totals) / min(train_totals)
    print(f'\ntrain imbalance ratio (max:min): {ratio:.1f}:1')
    if ratio > 5:
        print('  ⚠️  >5:1 — consider oversampling or class weights')

## 7. Source distribution per split

Drink Waste images are prefixed `dw_`. If one source dominates a particular split, your metrics will reflect that source, not real-world.

In [ ]:
for s in SPLITS:
    taco, dw = 0, 0
    for p in imgs_in(s):
        if p.stem.startswith('dw_'):
            dw += 1
        else:
            taco += 1
    total_s = taco + dw
    print(f'{s:>5}: TACO={taco:>5}  DrinkWaste={dw:>5}  (DW share: {(dw / total_s * 100) if total_s else 0:.0f}%)')

## 8. Image dimensions + aspect ratios

Ultralytics resizes to `imgsz` squarish. Extreme aspect ratios (>3:1) cause heavy letterboxing; unusually large originals mean more downsampling. Sample 300 for speed.

In [ ]:
candidates = imgs_in('train')
sample = random.sample(candidates, min(300, len(candidates)))
dims, ars = [], []
for p in sample:
    img = cv2.imread(str(p))
    if img is None:
        continue
    h, w = img.shape[:2]
    dims.append((w, h))
    ars.append(w / h)

widths = [d[0] for d in dims]
heights = [d[1] for d in dims]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(widths, heights, alpha=0.4, s=10)
axes[0].axvline(640, color='r', linestyle='--', alpha=0.5, label='imgsz=640')
axes[0].axhline(640, color='r', linestyle='--', alpha=0.5)
axes[0].axvline(960, color='g', linestyle='--', alpha=0.5, label='imgsz=960')
axes[0].axhline(960, color='g', linestyle='--', alpha=0.5)
axes[0].set_xlabel('width (px)')
axes[0].set_ylabel('height (px)')
axes[0].set_title(f'image dimensions (sample {len(dims)})')
axes[0].legend()

axes[1].hist(ars, bins=30)
axes[1].axvline(1.0, color='k', linestyle='--', alpha=0.5, label='1:1')
axes[1].axvline(3.0, color='r', linestyle='--', alpha=0.5, label='3:1 (extreme)')
axes[1].axvline(1/3, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('aspect ratio (w/h)')
axes[1].set_title('image aspect ratios')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f'median size: {int(np.median(widths))} × {int(np.median(heights))}')
print(f'width  5-95 pct: {int(np.percentile(widths, 5))} .. {int(np.percentile(widths, 95))}')
print(f'height 5-95 pct: {int(np.percentile(heights, 5))} .. {int(np.percentile(heights, 95))}')
print(f'extreme ARs (>3:1 or <1:3): {sum(1 for a in ars if a > 3 or a < 1/3)} / {len(ars)}')

## 9. Image color modes

YOLO expects 3-channel RGB. Grayscale images (L mode) or RGBA with transparency can slip through dataset prep. Ultralytics usually handles them but it's sloppy.

In [ ]:
modes = Counter()
sample = random.sample(imgs_in('train'), min(500, len(imgs_in('train'))))
for p in sample:
    try:
        with Image.open(p) as im:
            modes[im.mode] += 1
    except Exception:
        modes['unreadable'] += 1
for m, n in modes.most_common():
    print(f'{m:>10}: {n}')
if set(modes) - {'RGB'}:
    print('\n⚠️  non-RGB modes found — usually fine but worth noting')

## 10. Bbox size distribution

Normalized (fraction of image area). Boxes <2% area are hard at imgsz=640. If majority is <2%, bump imgsz.

In [ ]:
all_areas = []
per_class_areas = defaultdict(list)
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    for cls, cx, cy, w, h in parse_label(lbl):
        area = w * h * 100
        all_areas.append(area)
        per_class_areas[cls].append(area)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(all_areas, bins=50)
ax1.axvline(2, color='r', linestyle='--', label='2% (tiny)')
ax1.set_xlabel('bbox area (% of image)')
ax1.set_ylabel('count')
ax1.set_title('bbox area distribution (train)')
ax1.legend()

box_data = [per_class_areas[c] for c in range(len(CLASS_NAMES)) if per_class_areas[c]]
box_labels = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if per_class_areas[c]]
ax2.boxplot(box_data, tick_labels=box_labels)
ax2.set_yscale('log')
ax2.set_ylabel('bbox area (% of image, log)')
ax2.set_title('bbox area by class')
plt.tight_layout()
plt.show()

arr = np.array(all_areas)
print(f'boxes: {len(arr)}')
print(f'  median: {np.median(arr):.2f}%')
print(f'  5 pct:  {np.percentile(arr, 5):.2f}%')
print(f'  tiny (<2%): {(arr < 2).sum()} ({(arr < 2).mean() * 100:.0f}%)')

## 11. Bbox aspect ratios

Very elongated boxes (>5:1) break default anchor priors. Common with straws, flat wrappers, paper sheets. If a class has many extreme boxes, verify annotations weren't drawn around a pile of multiple objects.

In [ ]:
bbox_ars = defaultdict(list)
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    for cls, cx, cy, w, h in parse_label(lbl):
        if h > 0:
            bbox_ars[cls].append(w / h)

fig, ax = plt.subplots(figsize=(8, 4))
box_data = [bbox_ars[c] for c in range(len(CLASS_NAMES)) if bbox_ars[c]]
box_labels = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if bbox_ars[c]]
ax.boxplot(box_data, tick_labels=box_labels)
ax.axhline(1, color='k', linestyle='--', alpha=0.3, label='1:1')
ax.axhline(5, color='r', linestyle='--', alpha=0.5, label='5:1 (extreme)')
ax.set_yscale('log')
ax.set_ylabel('bbox w/h (log)')
ax.set_title('bbox aspect ratio by class')
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'class':<10} {'n':>6} {'median_ar':>10} {'>5:1 or <1:5':>14}")
for c, name in enumerate(CLASS_NAMES):
    ars_c = bbox_ars.get(c, [])
    if not ars_c:
        continue
    arr = np.array(ars_c)
    extreme = ((arr > 5) | (arr < 1/5)).sum()
    print(f'{name:<10} {len(arr):>6} {np.median(arr):>10.2f} {extreme:>14}')

## 12. Per-class bbox stats table

Single summary: how many instances, min/median/max bbox area, typical aspect ratio.

In [ ]:
print(f"{'class':<10} {'count':>6} {'min_area%':>10} {'med_area%':>10} {'max_area%':>10} {'med_ar':>8}")
for c, name in enumerate(CLASS_NAMES):
    areas = per_class_areas.get(c, [])
    ars_c = bbox_ars.get(c, [])
    if not areas:
        print(f'{name:<10} {0:>6}  (no samples)')
        continue
    a = np.array(areas)
    ar = np.array(ars_c) if ars_c else np.array([0])
    print(f'{name:<10} {len(a):>6} {a.min():>10.3f} {np.median(a):>10.3f} {a.max():>10.3f} {np.median(ar):>8.2f}')

## 13. Bbox center-position heatmap

2D histogram of bbox centers normalized to [0,1]×[0,1]. If all objects are dead-center the model may learn a positional prior that breaks in deployment (robot sees trash at the edge of frame during final approach).

In [ ]:
centers = defaultdict(list)
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    for cls, cx, cy, w, h in parse_label(lbl):
        centers[cls].append((cx, cy))

n_shown = sum(1 for c in range(len(CLASS_NAMES)) if centers[c])
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(3 * len(CLASS_NAMES), 3))
for c, name in enumerate(CLASS_NAMES):
    ax = axes[c]
    pts = centers.get(c, [])
    if pts:
        cxs, cys = zip(*pts)
        ax.hist2d(cxs, cys, bins=20, range=[[0, 1], [0, 1]], cmap='hot')
    ax.set_title(f'{name} (n={len(pts)})')
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)  # image coord convention
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
plt.suptitle('bbox center density (top-left = 0,0)')
plt.tight_layout()
plt.show()

## 14. Boxes per image

In [ ]:
boxes_per_img = []
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    boxes_per_img.append(len(parse_label(lbl)))

plt.figure(figsize=(7, 3))
plt.hist(boxes_per_img, bins=range(0, max(boxes_per_img) + 2))
plt.xlabel('boxes per image')
plt.ylabel('count')
plt.title('boxes per image (train)')
plt.show()
print(f'empty labels:  {sum(1 for n in boxes_per_img if n == 0)}')
print(f'median:        {int(np.median(boxes_per_img))}')
print(f'max:           {max(boxes_per_img)}')
print(f'p95:           {int(np.percentile(boxes_per_img, 95))}')

## 15. Class co-occurrence

Row = A, col = B: number of train images containing at least one of class A and at least one of class B. Diagonal = images containing the class. Off-diagonal = multi-class images.

In [ ]:
n = len(CLASS_NAMES)
mat = np.zeros((n, n), dtype=int)
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    classes = {e[0] for e in parse_label(lbl)}
    for a in classes:
        for b in classes:
            mat[a][b] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mat, cmap='Blues')
ax.set_xticks(range(n)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(n)); ax.set_yticklabels(CLASS_NAMES)
for i in range(n):
    for j in range(n):
        ax.text(j, i, int(mat[i][j]), ha='center', va='center', color='white' if mat[i][j] > mat.max() / 2 else 'black')
ax.set_title('co-occurrence (images containing both classes)')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 16. Sample gallery per class

4 random training images per class with ground-truth boxes drawn. Green = correct class; orange = other class present in same image.

In [ ]:
imgs_by_class = defaultdict(list)
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    classes = {e[0] for e in parse_label(lbl)}
    candidates = list((DATA / 'images' / 'train').glob(lbl.stem + '.*'))
    if not candidates:
        continue
    for c in classes:
        imgs_by_class[c].append((candidates[0], lbl))

random.seed(1)
cols = 4
fig, axes = plt.subplots(len(CLASS_NAMES), cols, figsize=(cols * 3, len(CLASS_NAMES) * 3))
for row, (c, name) in enumerate(zip(range(len(CLASS_NAMES)), CLASS_NAMES)):
    pool = imgs_by_class.get(c, [])
    samples = random.sample(pool, min(cols, len(pool)))
    for col in range(cols):
        ax = axes[row][col] if len(CLASS_NAMES) > 1 else axes[col]
        ax.axis('off')
        if col >= len(samples):
            continue
        img_path, lbl_path = samples[col]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        for cls, cx, cy, bw, bh in parse_label(lbl_path):
            x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
            x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
            color = (0, 200, 0) if cls == c else (255, 140, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, CLASS_NAMES[cls], (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        ax.imshow(img)
        if col == 0:
            ax.set_ylabel(name, rotation=0, ha='right', va='center', fontsize=12)
plt.tight_layout()
plt.show()

## 17. Health checks (label format)

Automated sanity on every label line: empty files, malformed, bad class IDs, out-of-range coordinates, degenerate boxes.

In [ ]:
issues = Counter()
examples = defaultdict(list)
for s in SPLITS:
    for lbl in (DATA / 'labels' / s).glob('*.txt'):
        text = lbl.read_text().strip()
        if not text:
            issues['empty_label'] += 1
            if len(examples['empty_label']) < 5:
                examples['empty_label'].append(lbl.name)
            continue
        for line in text.splitlines():
            parts = line.split()
            if len(parts) != 5:
                issues['malformed_line'] += 1
                continue
            try:
                cls, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
            except ValueError:
                issues['parse_error'] += 1
                continue
            if not (0 <= cls < len(CLASS_NAMES)):
                issues['bad_class_id'] += 1
                if len(examples['bad_class_id']) < 5:
                    examples['bad_class_id'].append((lbl.name, cls))
            if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                issues['coord_out_of_range'] += 1
                if len(examples['coord_out_of_range']) < 5:
                    examples['coord_out_of_range'].append(lbl.name)
            if w < 0.005 or h < 0.005:
                issues['degenerate_tiny_box'] += 1

if not issues:
    print('✅ no label format issues')
else:
    for k, v in issues.items():
        print(f'{k}: {v}')
        for ex in examples.get(k, []):
            print(f'  e.g. {ex}')

## Summary

If you got here with no red flags:
- File integrity clean → no mid-training crashes
- No cross-split duplicates → test metrics honest
- No orphans, no label format errors → Ultralytics will happily train
- Class distribution acceptable → no starved classes
- Bbox sizes manageable → imgsz choice validated
- Sample gallery looks correct → annotations actually match images

Ready to train.